# 🛰️ DEBRIS-SCAN AI — Cloud GPU Backend Server

This notebook runs the **FastAPI + Real AI Inference (YOLO11n + Radar DSP + Multimodal Fusion)** backend for the **DEBRIS-SCAN AI** Search & Rescue Mission Control system.

### 🚀 Quick Start (1 Click):
1. In the top menu, make sure GPU is enabled: **Runtime → Change runtime type → T4 GPU**.
2. Click **Runtime → Run all** (or press `Ctrl + F9`).
3. Scroll to the bottom cell — copy the generated **Public HTTPS URL** (e.g. `https://xxxx.trycloudflare.com`).
4. Open your live Vercel Mission Control: **[https://frontend-mu-fawn-39.vercel.app](https://frontend-mu-fawn-39.vercel.app)**
5. Paste the URL into the top-right backend box to connect from any device, anywhere in the world!

## 1. Clone Repository & Setup Working Directory

In [ ]:
# Remove old clone if re-running
!rm -rf Rescue-Drone
!git clone https://github.com/Atharva-Sharma7/Rescue-Drone.git
%cd Rescue-Drone/debris-scan-ai/backend
!pwd

## 2. Install AI & Backend Dependencies

In [ ]:
!pip install -q fastapi uvicorn websockets python-multipart scipy numpy ultralytics opencv-python-headless

import torch
print(f"✓ PyTorch {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU Device: {torch.cuda.get_device_name(0)}")

## 3. Launch Secure Public Tunnel & Start Mission Control API

In [ ]:
import subprocess
import time
import re
import sys

# 1. Download and setup cloudflared tunnel binary (zero-config, free HTTPS/WSS, no signup required)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# 2. Start FastAPI server in the background
server_proc = subprocess.Popen([sys.executable, "server.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(3)

# 3. Start Cloudflare Tunnel
tunnel_proc = subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:8000"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

public_url = None
for line in tunnel_proc.stderr:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print("=" * 70)
if public_url:
    print("🎉 DEBRIS-SCAN AI BACKEND IS LIVE ON GPU!")
    print(f"📡 Public Backend URL: {public_url}")
    print("=" * 70)
    print("\n👉 NEXT STEPS:")
    print(f"1. Open your Vercel Dashboard: https://frontend-mu-fawn-39.vercel.app")
    print(f"2. In the top-right backend input, paste: {public_url}")
    print("3. Connection indicator will turn GREEN! Click 'Create Mission' to run live AI inference!")
else:
    print("Could not retrieve tunnel URL. Check logs above.")
print("=" * 70)

# Stream server logs
try:
    for line in server_proc.stdout:
        print(line, end="")
except KeyboardInterrupt:
    print("\nStopping server...")
    server_proc.terminate()
    tunnel_proc.terminate()